In [2]:
import json
from pathlib import Path
import pandas as pd
from sklearn.metrics import f1_score

root = Path("results") if Path("results").exists() else Path("../results")
order = ["lyrics_clean", "lyrics_norm", "lyrics_masked", "lyrics_norm_masked"]

In [3]:
import json
from pathlib import Path
import pandas as pd

root = Path("results") if Path("results").exists() else Path("../results")

rows = []
for path in root.rglob("results.json"):
    parts = path.relative_to(root).parts   # e.g. ('ngram', 'Death', 'lyrics', 'results.json')
    if len(parts) < 4:                     # old runs saved before the text-version folders
        print("skipping", path)
        continue
    r = json.load(open(path))
    rows.append({"model": parts[0], "set": parts[1], "text": parts[2],
                 "accuracy": r["test_accuracy"], "macro_f1": r["test_macro_f1"]})

res = pd.DataFrame(rows)
print(len(res), "runs loaded")
res.groupby(["model", "set"])["text"].apply(sorted)

24 runs loaded


model  set  
bert   Death    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Mixed    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Power    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
ngram  Death    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Mixed    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
       Power    [lyrics_clean, lyrics_masked, lyrics_norm, lyr...
Name: text, dtype: object

In [4]:
order = ["lyrics_clean", "lyrics_norm", "lyrics_masked", "lyrics_norm_masked"]
table = res.pivot_table(index=["model", "set"], columns="text", values="macro_f1")[order]
display(table.round(3))
display(table.sub(table["lyrics_clean"], axis=0).round(3))   # change from baseline

text         lyrics_clean  lyrics_norm  lyrics_masked  lyrics_norm_masked
model set                                                                
bert  Death         0.627        0.671          0.623               0.610
      Mixed         0.598        0.525          0.426               0.522
      Power         0.609        0.552          0.553               0.524
ngram Death         0.802        0.820          0.792               0.790
      Mixed         0.737        0.692          0.722               0.650
      Power         0.780        0.744          0.750               0.724

text         lyrics_clean  lyrics_norm  lyrics_masked  lyrics_norm_masked
model set                                                                
bert  Death           0.0        0.044         -0.004              -0.017
      Mixed           0.0       -0.073         -0.173              -0.077
      Power           0.0       -0.058         -0.057              -0.085
ngram Death           0.0        0.018         -0.010              -0.012
      Mixed           0.0       -0.045         -0.015              -0.086
      Power           0.0       -0.036         -0.030              -0.055

In [5]:
order = ["lyrics_clean", "lyrics_norm", "lyrics_masked", "lyrics_norm_masked"]

def band_f1(model, set_, text):
    """Per-band F1 for one run, read from its predictions.csv."""
    files = list((root / model / set_ / text).rglob("predictions.csv"))
    if not files:
        print(f"no predictions.csv for {model}/{set_}/{text}")
        return None
    p = pd.read_csv(files[0])
    bands = sorted(p["true"].unique())
    return pd.Series(f1_score(p["true"], p["pred"], labels=bands, average=None),
                     index=bands)

def band_table(model, set_):
    cols = {t: band_f1(model, set_, t) for t in order}
    t = pd.DataFrame({k: v for k, v in cols.items() if v is not None})
    base = t["lyrics_clean"]
    for c in t.columns.drop("lyrics_clean"):
        t[f"Δ {c.replace('lyrics_', '')}"] = t[c] - base
    return t.round(2)

band_table("ngram", "Mixed").sort_values("Δ norm")

,lyrics_clean,lyrics_norm,lyrics_masked,lyrics_norm_masked,Δ norm,Δ masked,Δ norm_masked
Accept,0.76,0.46,0.79,0.44,-0.31,0.02,-0.32
Iron Maiden,0.63,0.49,0.57,0.48,-0.15,-0.06,-0.15
Sevendust,0.78,0.71,0.75,0.67,-0.07,-0.03,-0.11
King Diamond,0.98,0.91,0.98,0.91,-0.07,0.00,-0.07
Black Label Society,0.73,0.71,0.71,0.67,-0.02,-0.02,-0.07
Venom,0.74,0.72,0.65,0.64,-0.02,-0.09,-0.10
GWAR,0.71,0.71,0.71,0.61,0.00,0.00,-0.09
W.A.S.P.,0.75,0.77,0.68,0.75,0.02,-0.07,0.00
Korn,0.75,0.78,0.68,0.67,0.03,-0.07,-0.08
Megadeth,0.54,0.67,0.69,0.67,0.13,0.15,0.13


In [7]:
for model in ["ngram", "bert"]:
    for s in ["Death", "Power", "Mixed"]:
        print(f"\n=== {model} / {s} ===")
        display(band_table(model, s))


=== ngram / Death ===


,lyrics_clean,lyrics_norm,lyrics_masked,lyrics_norm_masked,Δ norm,Δ masked,Δ norm_masked
Cannibal Corpse,0.80,0.86,0.79,0.83,0.06,-0.01,0.03
Deicide,0.89,0.93,0.94,0.91,0.04,0.05,0.02
Immolation,0.92,0.88,0.90,0.83,-0.05,-0.02,-0.09
Incantation,0.76,0.82,0.81,0.82,0.06,0.05,0.06
Krisiun,0.86,0.90,0.84,0.86,0.04,-0.03,-0.01
Malevolent Creation,0.69,0.65,0.69,0.69,-0.04,0.00,-0.00
Napalm Death,0.89,0.86,0.85,0.83,-0.03,-0.04,-0.07
Six Feet Under,0.84,0.84,0.77,0.80,0.00,-0.07,-0.04
Unleashed,0.80,0.80,0.78,0.76,0.00,-0.02,-0.04
Vader,0.56,0.65,0.56,0.58,0.09,0.00,0.02



=== ngram / Power ===


,lyrics_clean,lyrics_norm,lyrics_masked,lyrics_norm_masked,Δ norm,Δ masked,Δ norm_masked
Blind Guardian,0.89,0.89,0.87,0.89,0.00,-0.02,0.00
Bloodbound,0.74,0.74,0.77,0.73,0.00,0.03,-0.02
Elvenking,0.88,0.88,0.88,0.82,0.00,0.00,-0.06
Freedom Call,0.93,0.90,0.95,0.88,-0.02,0.02,-0.05
Grave Digger,0.82,0.77,0.70,0.72,-0.05,-0.12,-0.10
HammerFall,0.76,0.70,0.67,0.68,-0.05,-0.09,-0.07
Helloween,0.59,0.55,0.63,0.60,-0.04,0.05,0.01
Iced Earth,0.74,0.71,0.74,0.67,-0.04,0.00,-0.08
Kamelot,0.72,0.62,0.59,0.60,-0.10,-0.13,-0.12
Rage,0.73,0.67,0.70,0.65,-0.06,-0.03,-0.08



=== ngram / Mixed ===


,lyrics_clean,lyrics_norm,lyrics_masked,lyrics_norm_masked,Δ norm,Δ masked,Δ norm_masked
Accept,0.76,0.46,0.79,0.44,-0.31,0.02,-0.32
Black Label Society,0.73,0.71,0.71,0.67,-0.02,-0.02,-0.07
GWAR,0.71,0.71,0.71,0.61,0.00,0.00,-0.09
Iron Maiden,0.63,0.49,0.57,0.48,-0.15,-0.06,-0.15
King Diamond,0.98,0.91,0.98,0.91,-0.07,0.00,-0.07
Korn,0.75,0.78,0.68,0.67,0.03,-0.07,-0.08
Megadeth,0.54,0.67,0.69,0.67,0.13,0.15,0.13
Sevendust,0.78,0.71,0.75,0.67,-0.07,-0.03,-0.11
Venom,0.74,0.72,0.65,0.64,-0.02,-0.09,-0.10
W.A.S.P.,0.75,0.77,0.68,0.75,0.02,-0.07,0.00


... (2 outputs left out)


=== bert / Power ===


,lyrics_clean,lyrics_norm,lyrics_masked,lyrics_norm_masked,Δ norm,Δ masked,Δ norm_masked
Blind Guardian,0.80,0.67,0.82,0.77,-0.13,0.02,-0.03
Bloodbound,0.68,0.59,0.60,0.60,-0.09,-0.08,-0.08
Elvenking,0.67,0.65,0.63,0.63,-0.02,-0.04,-0.04
Freedom Call,0.72,0.72,0.69,0.70,0.00,-0.02,-0.02
Grave Digger,0.67,0.40,0.47,0.47,-0.27,-0.20,-0.20
HammerFall,0.65,0.55,0.53,0.51,-0.10,-0.12,-0.14
Helloween,0.44,0.45,0.30,0.29,0.01,-0.14,-0.15
Iced Earth,0.54,0.56,0.56,0.49,0.02,0.02,-0.05
Kamelot,0.39,0.36,0.40,0.28,-0.02,0.01,-0.11
Rage,0.55,0.56,0.53,0.51,0.01,-0.02,-0.04



=== bert / Mixed ===


,lyrics_clean,lyrics_norm,lyrics_masked,lyrics_norm_masked,Δ norm,Δ masked,Δ norm_masked
Accept,0.47,0.37,0.30,0.30,-0.11,-0.17,-0.17
Black Label Society,0.71,0.59,0.47,0.68,-0.13,-0.25,-0.03
GWAR,0.56,0.41,0.47,0.35,-0.14,-0.08,-0.20
Iron Maiden,0.44,0.41,0.40,0.50,-0.03,-0.04,0.06
King Diamond,0.89,0.83,0.82,0.82,-0.06,-0.07,-0.07
Korn,0.50,0.51,0.21,0.40,0.01,-0.29,-0.10
Megadeth,0.42,0.29,0.00,0.25,-0.14,-0.42,-0.17
Sevendust,0.74,0.63,0.61,0.63,-0.11,-0.13,-0.11
Venom,0.72,0.65,0.58,0.72,-0.07,-0.13,0.00
W.A.S.P.,0.53,0.57,0.39,0.56,0.05,-0.14,0.04
